# 🔍 Projeto Íris - Web Scraper Abrangente de Vagas de Emprego

Este notebook realiza scraping de 3 Sites de Vaga de Emprego no Brasil:

### Sites Generalistas:
- **Catho** - Um dos maiores portais de emprego
- **InfoJobs** - Portal consolidado de vagas


### Sites de Classificados:
- **OLX** - Classificados gerais

---

### Autores
João Alex (SSIT) e Pedro Henrique (IEEE CIS)

## 📦 Instalação de Dependências

In [ ]:
# Instalação das bibliotecas necessárias
!pip install requests beautifulsoup4 pandas lxml selenium fake-useragent cloudscraper

: 

## 🛠️ Imports e Configuração

In [3]:
import requests
import pandas as pd
from urllib.parse import quote, urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import time
import random
from datetime import datetime
from fake_useragent import UserAgent
import warnings
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
warnings.filterwarnings('ignore')

# Configuração de Logs
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger("IrisScraper")

# User Agent dinâmico
ua = UserAgent()

def get_headers():
    """Gera headers dinâmicos para evitar bloqueios"""
    return {
        'User-Agent': ua.random,
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept-Encoding': 'gzip, deflate, br',
        'DNT': '1',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'
    }

def safe_request(url, max_retries=3):
    """Faz requisição com retry automático"""
    for attempt in range(max_retries):
        try:
            time.sleep(random.uniform(1, 3))  # Rate limiting
            response = requests.get(url, headers=get_headers(), timeout=15, verify=False)
            response.raise_for_status()
            return response
        except Exception as e:
            if attempt == max_retries - 1:
                logger.warning(f"Falha após {max_retries} tentativas: {url}")
                return None
            time.sleep(random.uniform(2, 5))
    return None

## 🌐 Scrapers por Site

### 1. CATHO

In [ ]:
from time import sleep


def scrape_catho(keyword):
    """Scraper para Catho.com.br"""
    source_name = "Catho"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        # URL de busca
        keyword_encoded = quote(keyword)
        url = f"https://www.catho.com.br/vagas/{keyword_encoded}/"

        driver = webdriver.Firefox()
        driver.get(url)

        page = 1
        max_pages = -1 # will fetch this as soon as it reaches the end of the page
        
        if page == 1:
            #Scrollar um pouco a tela para triggar popup de ativação de cookies
            driver.execute_script("window.scrollTo(0,50)")
            #Aceitar Cookies
            WebDriverWait(driver, 60).until(EC.element_to_be_clickable((By.XPATH, "//*[text() = 'Aceitar todos os cookies' ]"))).click()
        else:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "article")))

        while page != max_pages + 1:

            #Carregar toda a página antes
            last_height = driver.execute_script("return document.documentElement.scrollHeight")
            while True:
                driver.execute_script("window.scrollTo(0,document.documentElement.scrollHeight);")
                time.sleep(3)
                new_height = driver.execute_script("return document.documentElement.scrollHeight")
                if new_height == last_height:
                    break
                last_height = new_height
            
            job_cards = driver.find_elements(By.TAG_NAME, "article")
            for card in job_cards:
                try:
                    #Focar Elemento
                    driver.execute_script("arguments[0].scrollIntoView();", card)            
                    card.click()
                    sleep(1.5)

                    #Título da Vaga
                    title_elem = card.find_element(By.TAG_NAME, "h2") or card.find_element(By.TAG_NAME, "h3")
                    if not title_elem:
                        continue

                    title = title_elem.text

                    #Link da Vaga
                    link_elem = card.find_element(By.TAG_NAME, 'a')
                    href = link_elem.get_attribute('href')

                    link = urljoin('https://www.catho.com.br', href) if link_elem else url
                    
                    # Empresa
                    company_elem = driver.find_element(By.CLASS_NAME, 'text-neutral')
                    company = company_elem.text if company_elem.text else "Não informado"    
                    
                    salary_job_quantity = card.find_elements(By.TAG_NAME, 'strong')
                    salary = ""
                    quantity = ""
                    # Qtd Vagas e Salário
                    if "R$" in salary_job_quantity[0].text:
                        quantity = salary_job_quantity[1].text
                        salary = salary_job_quantity[0].text
                    else:
                        quantity = salary_job_quantity[0].text
                        salary = salary_job_quantity[1].text                 

                    # Localização
                    location_elem = card.find_element(By.TAG_NAME, 'strong').find_element(By.XPATH, "..")
                    location = location_elem.text if location_elem.text else "Brasil"

                    #Descricao
                    description_elem = driver.find_element(By.CLASS_NAME, "whitespace-pre-line")
                    descrpition = description_elem.text if description_elem.text else "Não informado"
                    
                    vagas.append({
                        'Fonte': source_name,
                        'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                        'Cargo': title,
                        'Qtd Vagas': quantity,
                        'Descrição': descrpition,
                        'Empresa': company,
                        'Local': location,
                        'Salário': salary,
                        'Link': link
                    })
                except Exception as e:
                    continue

            if max_pages == -1:
                max_page_elem = driver.find_element(By.XPATH, "//*[@class='page-button buildLink ']") 
                print(max_page_elem.text)
                max_pages = int(max_page_elem.text)
                if page == max_pages: 
                    break
            page += 1
            next_page_elem = driver.find_element(By.CLASS_NAME, "next-page")
            next_page_elem.click()

    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas
    

### 2. INFOJOBS

In [ ]:
def scrape_infojobs(keyword):
    """Scraper para InfoJobs.com.br"""
    source_name = "InfoJobs"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword.replace(' ', '+'))
        url = f"https://www.infojobs.com.br/vagas-de-emprego-{keyword_encoded}.aspx"
        
        driver =  webdriver.Firefox()
        driver.get(url)
        
        #Aceitar cookies e remover Pop-Up
        WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, "//*[ text() = 'Aceitar' ]"))).click()
        WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, "//*[ contains(text(),'Agora não')]"))).click()
        
        
        #Carregar toda a página antes
        last_height = driver.execute_script("return document.documentElement.scrollHeight")
        while True:
            driver.execute_script("window.scrollTo(0,document.documentElement.scrollHeight);")
            time.sleep(3)
            new_height = driver.execute_script("return document.documentElement.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        # Buscar vagas
        job_elements = driver.find_elements(By.XPATH, "//*[@class='pt-24 px-24 cursor-pointer js_vacancyLoad js_rowCard js_cardLink']") 
        
        for job in job_elements:
            try:
                #Focar Elemento
                driver.execute_script("arguments[0].scrollIntoView();", job)            
                job.click()
                time.sleep(3)

                title_elem = job.find_element(By.TAG_NAME, 'h2')
                if not title_elem:
                    continue
                    
                title = title_elem.text

                #Link da Vaga
                link_elem = job.find_element(By.TAG_NAME, 'a')
                href = link_elem.get_attribute('href')

                link = urljoin('https://www.infojobs.com.br', href) if link_elem else url      

                
                # Localização e data
                location_elem = job.find_element(By.CLASS_NAME, 'mb-8')
                location = location_elem.text if location_elem else "Brasil"

                company_elem = driver.find_element(By.CLASS_NAME, "h4")
                company = company_elem.text

                description_elem = driver.find_element(By.XPATH, "//*[@class='mb-16 text-break white-space-pre-line']") 
                description = description_elem.text

                salary = job.find_element(By.XPATH, "//*[@class='icon icon-money   icon-size-16']").find_element(By.XPATH,"./..").text

                job_count_elem = driver.find_element(By.XPATH, "//*[ contains(text(),'Número de vagas')]").find_element(By.XPATH,"./..")

                job_count = job_count_elem.text

                
                vagas.append({
                        'Fonte': source_name,
                        'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                        'Cargo': title,
                        'Qtd Vagas': job_count,
                        'Descrição': description,
                        'Empresa': company,
                        'Local': location,
                        'Salário': salary,
                        'Link': link
                })
            except Exception as e:
                print(e)
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 3. OLX

In [14]:

import time
from selenium import webdriver
def scrape_olx(keyword):
    """Scraper para Olx.com.br"""
    source_name = "Olx"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    try:
        formatted_keyword = quote(keyword.replace(' ', '+'))
        url = f"https://www.olx.com.br/vagas-de-emprego/estado-df/distrito-federal-e-regiao?q={formatted_keyword}"

        driver = webdriver.Firefox()
        driver.get(url)
        original_window = driver.current_window_handle

        #Remover Anúncio
        driver.execute_script("window.scrollTo(0,document.documentElement.scrollHeight);")
        try:
            WebDriverWait(driver,60).until(EC.element_to_be_clickable((By.CLASS_NAME, "AdvertisingAnchor-module-scss-module__OnvCGG__anchor-container__close-button"))).click()
            WebDriverWait(driver,60).until(EC.element_to_be_clickable((By.CLASS_NAME, "adopt-c-GYsVN"))).click()
            
        except:
            pass
        job_card = driver.find_elements(By.XPATH, "//*[@class='olx-adcard__link']")

        number_win = 1
        for job in job_card:
            driver.execute_script("arguments[0].scrollIntoView();", job)   
            job.click()
            number_win += 1
            WebDriverWait(driver,10).until(EC.number_of_windows_to_be(number_win))
            breakable = False
            for win in driver.window_handles:
                driver.switch_to.window(win)
                if 'Cloudflare' in driver.title:
                    breakable = True
                    break
            driver.switch_to.window(original_window)
            if breakable:
                break
        for win in driver.window_handles:
            if win == original_window: 
                continue
            try:
                    driver.switch_to.window(win)
                    #Ad
                    WebDriverWait(driver,60).until(EC.element_to_be_clickable((By.XPATH, "//*[@class='ad__sc-hq36pb-1 gvCmGZ']"))).click()
                    link = driver.current_url
                    title = WebDriverWait(driver,60).until(EC.presence_of_element_located((By.XPATH, "//*[@class='typo-title-medium ad__sc-1l883pa-2 bdcWAn']"))).text
                    description = WebDriverWait(driver,60).until(EC.presence_of_element_located((By.XPATH, "//*[@style='word-break:break-word;white-space:break-spaces']"))).text
                    company = WebDriverWait(driver,60).until(EC.presence_of_element_located((By.XPATH, "//*[@class='typo-body-large ad__sc-ypp2u2-4 ftwmRA']"))).text
                    company_link = WebDriverWait(driver,60).until(EC.presence_of_element_located((By.XPATH, "//*[@class='olx-core-button olx-core-button--neutral olx-core-button--small w-full ad__sc-1bqzobc-2 fXTbOn']"))).get_attribute('href')
                    image_element =  driver.find_element(By.CLASS_NAME, "advc-gallery__root")
                    image_link = ''
                    try:
                        image_link = image_element.find_element(By.TAG_NAME, "img").get_attribute('src')
                    except:
                        logger.error("Não há imagem para a vaga. Prosseguindo...")
                    vagas.append({
                                            'Fonte': source_name,
                                            'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                                            'Cargo': title,
                                            'Descrição': description,
                                            'Empresa': company,
                                            'Link da Empresa': company_link,
                                            'Local': 'DF',
                                            'Link': link,
                                            'Imagem da Vaga': image_link
                                    })         
            except Exception as e:
                 logger.error(f"❌ [{source_name}] Erro: {e}")
                 continue   
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        driver.quit()
    driver.quit()
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas


## 🚀 Orquestrador Principal

In [15]:
def executar_pipeline_completo(keyword, max_workers=8):
    """
    Executa TODOS os scrapers em paralelo
    
    Args:
        keyword: Termo de busca (ex: 'cientista de dados', 'babá', 'modelo')
        max_workers: Número de scrapers simultâneos
    
    Returns:
        DataFrame com todas as vagas encontradas
    """
    logger.info("="*80)
    logger.info(f"🚀 INICIANDO PIPELINE COMPLETO - Busca: '{keyword}'")
    logger.info("="*80)
    
    # Lista de todos os scrapers
    scrapers = [
        #scrape_catho,
        #scrape_infojobs,
        scrape_olx,
    ]
    
    todos_resultados = []
    
    # Execução paralela
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Agendar todas as buscas
        futures = {executor.submit(scraper, keyword): scraper.__name__ for scraper in scrapers}
        
        # Processar resultados conforme completam
        for future in as_completed(futures):
            scraper_name = futures[future]
            try:
                resultado = future.result()
                if resultado:
                    todos_resultados.extend(resultado)
                    logger.info(f"✓ {scraper_name} completado")
            except Exception as e:
                logger.error(f"✗ Erro em {scraper_name}: {e}")
    
    # Criar DataFrame
    if not todos_resultados:
        logger.error("❌ Nenhuma vaga encontrada em nenhum site.")
        return pd.DataFrame()
    
    df = pd.DataFrame(todos_resultados)
    
    # Remover duplicatas baseadas no Link
    df.drop_duplicates(subset=['Link'], inplace=True)
    
    # Estatísticas por fonte
    logger.info("="*80)
    logger.info("📊 ESTATÍSTICAS POR FONTE:")
    stats = df['Fonte'].value_counts()
    for fonte, count in stats.items():
        logger.info(f"   {fonte}: {count} vagas")
    
    logger.info("="*80)
    logger.info(f"🏁 TOTAL CONSOLIDADO: {len(df)} vagas únicas encontradas")
    logger.info("="*80)
    
    return df

## 🔍 Busca Avançada - Múltiplos Termos

Para o Projeto Íris, podemos buscar múltiplos termos de risco:

In [16]:
# BUSCA MÚLTIPLA - Para análise de risco do Projeto Íris
termos_risco = [
    'babá',
    'modelo',
    'recepcionista',
    'promotora',
    'garçonete',
    'atendente',
    'cuidadora',
    'secretária',
    'serviços gerais',
    'doméstica',
    'faxineira'
]

print("🔍 INICIANDO BUSCA MÚLTIPLA PARA ANÁLISE DE RISCO")
print(f"Termos a buscar: {', '.join(termos_risco)}")
print("="*80)

# Coletar todas as vagas
df_todos_termos = pd.DataFrame()

for termo in termos_risco:
    print(f"\n🔎 Buscando: {termo}")
    df_temp = executar_pipeline_completo(termo, max_workers=4)
    if not df_temp.empty:
        df_temp['Termo_Busca'] = termo
        df_todos_termos = pd.concat([df_todos_termos, df_temp], ignore_index=True)
    print(f"   Encontradas: {len(df_temp)} vagas")
   
    time.sleep(1)  # Pausa entre buscas

# Remover duplicatas globais
df_todos_termos.drop_duplicates(subset=['Link'], inplace=True)

# Salvar dataset consolidado
if not df_todos_termos.empty:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename_consolidado = f'dataset_iris_completo_{timestamp}.csv'
    df_todos_termos.to_csv(filename_consolidado, index=False, encoding='utf-8-sig', sep='~')
    
    print("\n" + "="*80)
    print("✅ BUSCA MÚLTIPLA CONCLUÍDA!")
    print(f"Total de vagas coletadas: {len(df_todos_termos)}")
    print(f"Arquivo salvo: {filename_consolidado}")
    print("\nDistribuição por termo de busca:")
    print(df_todos_termos['Termo_Busca'].value_counts())
    print("="*80)

20:02:20 - ================================================================================
20:02:20 - 🚀 INICIANDO PIPELINE COMPLETO - Busca: 'babá'
20:02:20 - ================================================================================
20:02:21 - 🕷️ [Olx] Iniciando busca...


🔍 INICIANDO BUSCA MÚLTIPLA PARA ANÁLISE DE RISCO
Termos a buscar: babá, modelo, recepcionista, promotora, garçonete, atendente, cuidadora, secretária, serviços gerais, doméstica, faxineira

🔎 Buscando: babá


20:04:40 - ❌ [Olx] Erro: Message: 
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:169:5
NoSuchElementError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:538:5
dom.find/</<@chrome://remote/content/shared/DOM.sys.mjs:137:16

20:05:40 - ❌ [Olx] Erro: Message: 
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:169:5
NoSuchElementError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:538:5
dom.find/</<@chrome://remote/content/shared/DOM.sys.mjs:137:16

20:07:49 - ✅ [Olx] 9 vagas encontradas.
20:07:49 - ✓ scrape_olx completado
20:07:54 - ================================================================================
20:07:54 - 📊 ESTATÍSTICAS POR FONTE:
20:07:55 -    Olx: 9 vagas
20:07:55 - ================================================================================
20:07

   Encontradas: 9 vagas


20:07:57 - ================================================================================
20:07:57 - 🚀 INICIANDO PIPELINE COMPLETO - Busca: 'modelo'
20:07:57 - ================================================================================
20:07:57 - 🕷️ [Olx] Iniciando busca...


...

   Encontradas: 3 vagas


20:48:55 - ================================================================================
20:48:55 - 🚀 INICIANDO PIPELINE COMPLETO - Busca: 'faxineira'
20:48:55 - ================================================================================
20:48:55 - 🕷️ [Olx] Iniciando busca...



🔎 Buscando: faxineira


20:50:29 - ✅ [Olx] 3 vagas encontradas.
20:50:29 - ✓ scrape_olx completado
20:50:29 - ================================================================================
20:50:29 - 📊 ESTATÍSTICAS POR FONTE:
20:50:29 -    Olx: 3 vagas
20:50:29 - ================================================================================
20:50:29 - 🏁 TOTAL CONSOLIDADO: 3 vagas únicas encontradas
20:50:29 - ================================================================================


   Encontradas: 3 vagas

✅ BUSCA MÚLTIPLA CONCLUÍDA!
Total de vagas coletadas: 65
Arquivo salvo: dataset_iris_completo_20260906_205030.csv

Distribuição por termo de busca:
Termo_Busca
recepcionista      10
serviços gerais    10
babá                9
cuidadora           9
garçonete           7
atendente           6
promotora           4
secretária          4
doméstica           3
faxineira           3
Name: count, dtype: int64
